In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

#Oil consumption table

url_consumption = 'https://www.worldometers.info/oil/oil-consumption-by-country/'

response_consumption = requests.get(url_consumption)
html_consumption = response_consumption.text

soup_consumption = BeautifulSoup(html_consumption, 'html.parser')
table_consumption = soup_consumption.find('table', {'class': 'datatable'})

headers_consumption = [th.text.strip() for th in table_consumption.find('thead').find_all('th')]

rows_consumption = []
for tr in table_consumption.find('tbody').find_all('tr'):
    row = [td.text.strip() for td in tr.find_all('td')]
    rows_consumption.append(row)

df_consumption = pd.DataFrame(rows_consumption, columns=headers_consumption)

print("Oil consumption data:")
display(df_consumption.head())

Oil consumption data:


,#,Country,Daily Oil Consumption (barrels),World Share,Yearly Gallons Per Capita
0,1,United States,"20,463,721",19.95%,908.2
1,2,China,"16,370,536",15.96%,176.8
2,3,India,"5,598,890",5.46%,59.2
3,4,Russia,"3,791,146",3.70%,401.3
4,5,Saudi Arabia,"3,630,712",3.54%,1638.8


In [ ]:
#Oil reserve table
url_reserves = 'https://www.worldometers.info/oil/'

response_reserves = requests.get(url_reserves)
html_reserves = response_reserves.text

soup_reserves = BeautifulSoup(html_reserves, 'html.parser')

df_reserves = None

table_reserves = soup_reserves.find('table', {'class': 'datatable'})

if table_reserves:

    headers_reserves = [th.text.strip() for th in table_reserves.find('thead').find_all('th')]


    rows_reserves = []
    for tr in table_reserves.find('tbody').find_all('tr'):
        row = [td.text.strip() for td in tr.find_all('td')]
        rows_reserves.append(row)


    df_reserves = pd.DataFrame(rows_reserves, columns=headers_reserves)

    print("\nOil Reseerves Data:")
    display(df_reserves.head())





Oil Reseerves Data:


,#,Country,Oil Reserves (barrels) in 2025,World Share
0,1,Venezuela,"303,008,000,000",17.17%
1,2,Saudi Arabia,"267,230,000,000",15.14%
2,3,Iran,"208,600,000,000",11.82%
3,4,Canada,"163,108,000,000",9.24%
4,5,Iraq,"145,019,000,000",8.22%


In [ ]:
#Merged table

df_merged = pd.merge(df_consumption, df_reserves, on='Country', how='inner')
print("Merged Oil Consumption and Reserves Data:")
display(df_merged.head())



Merged Oil Consumption and Reserves Data:


,#_x,Country,Daily Oil Consumption (barrels),World Share_x,Yearly Gallons Per Capita,#_y,Oil Reserves (barrels) in 2025,World Share_y
0,1,United States,"20,463,721",19.95%,908.2,8,"83,729,430,000",4.74%
1,2,China,"16,370,536",15.96%,176.8,13,"28,182,000,000",1.60%
2,3,India,"5,598,890",5.46%,59.2,23,"4,980,857,000",0.28%
3,4,Russia,"3,791,146",3.70%,401.3,9,"80,000,000,000",4.53%
4,5,Saudi Arabia,"3,630,712",3.54%,1638.8,2,"267,230,000,000",15.14%


In [ ]:
#Reserve to consumption ratio
df_merged['Daily Oil Consumption (barrels)'] = df_merged['Daily Oil Consumption (barrels)'].str.replace(',', '').astype(float)
df_merged['Oil Reserves (barrels) in 2025'] = df_merged['Oil Reserves (barrels) in 2025'].str.replace(',', '').astype(float)

df_merged['World Share_x'] = df_merged['World Share_x'].str.replace('%', '').astype(float) / 100
df_merged['World Share_y'] = df_merged['World Share_y'].str.replace('%', '').astype(float) / 100

df_merged.rename(columns={'World Share_x': 'Consumption World Share', 'World Share_y': 'Reserves World Share'}, inplace=True)

df_merged['Annual Oil Consumption (barrels)'] = df_merged['Daily Oil Consumption (barrels)'] * 365
df_merged['Reserve-to-Consumption Ratio'] = df_merged['Oil Reserves (barrels) in 2025'] / df_merged['Annual Oil Consumption (barrels)']

print("Updated DataFrame:")
display(df_merged.head())

df_merged.to_csv('Global_oil_reserves_vs_consumption_data.csv', index=False)



Updated DataFrame:


,#_x,Country,Daily Oil Consumption (barrels),Consumption World Share,Yearly Gallons Per Capita,#_y,Oil Reserves (barrels) in 2025,Reserves World Share,Annual Oil Consumption (barrels),Reserve-to-Consumption Ratio
0,1,United States,20463721.0,0.1995,908.2,8,8.372943e+10,0.0474,7.469258e+09,11.209872
1,2,China,16370536.0,0.1596,176.8,13,2.818200e+10,0.0160,5.975246e+09,4.716459
2,3,India,5598890.0,0.0546,59.2,23,4.980857e+09,0.0028,2.043595e+09,2.437302
3,4,Russia,3791146.0,0.0370,401.3,9,8.000000e+10,0.0453,1.383768e+09,57.813147
4,5,Saudi Arabia,3630712.0,0.0354,1638.8,2,2.672300e+11,0.1514,1.325210e+09,201.651077
